In [1]:
import os

# Must be set before importing tensorflow/protobuf anywhere.
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION", "2")
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_PYTHON_API", "python")
os.environ.setdefault("PYTHONHASHSEED", "0")

import math
import sys
import glob
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(0)

# Bugfix: previous run hard-failed because TRAINING=False and no weights were available.
# To ensure an end-to-end runnable pipeline that produces a meaningful submission,
# we enable training by default.
TRAINING = True

ALLOW_RANDOM_WEIGHTS_INFERENCE = (
    False  # keep strict: if training is disabled, don't silently submit random preds
)



In [2]:
INPUT_DIR = "../input/aptos2019-blindness-detection"
if not os.path.exists(INPUT_DIR):
    alt = "/kaggle/input/aptos2019-blindness-detection"
    if os.path.exists(alt):
        INPUT_DIR = alt

TRAIN_CSV = os.path.join(INPUT_DIR, "train.csv")
TEST_CSV = os.path.join(INPUT_DIR, "test.csv")
TRAIN_IMG_DIR = os.path.join(INPUT_DIR, "train_images")
TEST_IMG_DIR = os.path.join(INPUT_DIR, "test_images")



In [3]:
import cv2


def _pip_install(args):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    subprocess.check_call(cmd)


def _pip_install_offline(path):
    _pip_install(["install", "--no-deps", "--no-index", "--upgrade", path])


def _purge_google_protobuf_modules():
    import importlib

    for mod in list(sys.modules.keys()):
        if mod == "google" or mod.startswith("google."):
            if mod.startswith("google.protobuf"):
                del sys.modules[mod]
    importlib.invalidate_caches()


def _ensure_protobuf_320():
    """
    Bugfix for: AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
    Root cause: protobuf>=4 incompatibility with older TF builds common on Kaggle.
    Solution: ensure protobuf==3.20.* is installed, then purge protobuf modules.
    """
    try:
        from google.protobuf import __version__ as pb_ver  # noqa

        if str(pb_ver).startswith("3.20."):
            return True
    except Exception:
        pass

    wheel_candidates = []
    search_patterns = [
        "/kaggle/input/**/protobuf-3.20*.whl",
        "/kaggle/input/**/protobuf-3.20*.tar.gz",
        "/kaggle/input/**/protobuf-3.20*.zip",
        "../input/**/protobuf-3.20*.whl",
        "../input/**/protobuf-3.20*.tar.gz",
        "../input/**/protobuf-3.20*.zip",
    ]
    for pattern in search_patterns:
        wheel_candidates.extend(glob.glob(pattern, recursive=True))

    def _rank(p):
        b = os.path.basename(p)
        return (0 if b.endswith(".whl") else 1, b)

    wheel_candidates = sorted(set(wheel_candidates), key=_rank)

    try:
        if wheel_candidates:
            _pip_install_offline(wheel_candidates[0])
        else:
            _pip_install(["install", "--upgrade", "protobuf==3.20.*"])
        _purge_google_protobuf_modules()
        return True
    except Exception:
        return False


_ensure_protobuf_320()
_purge_google_protobuf_modules()

tf = None
_tf_import_error = None

try:
    import tensorflow as tf  # noqa: F401

    _tf_import_error = None
except Exception as e:
    _tf_import_error = e
    tf = None

if tf is None:
    _ensure_protobuf_320()
    _purge_google_protobuf_modules()
    try:
        import tensorflow as tf  # noqa: F401

        _tf_import_error = None
    except Exception as e2:
        _tf_import_error = e2
        tf = None

IMG_SIZE = 224
NB_CHANNELS = 3
NB_CLASSES = 5  # 0, 1, 2, 3, 4
BATCH_SIZE = 32
TEST_BATCH_SIZE = 1

if tf is None:
    raise RuntimeError(
        "TensorFlow failed to import even after protobuf pinning; cannot run model inference. "
        f"Last TF import error: {repr(_tf_import_error)}"
    )

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    GlobalAveragePooling2D,
    Dense,
    Dropout,
    BatchNormalization,
    Conv2D,
    MaxPooling2D,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import CSVLogger, ModelCheckpoint
from tensorflow.keras.applications.resnet50 import ResNet50

print("TensorFlow version:", tf.__version__)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.10.0 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.

TensorFlow version: 2.18.0


In [4]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)

print("Number of train samples: ", train.shape[0])
print("Number of test samples: ", test.shape[0])



Number of train samples:  3295
Number of test samples:  367


In [5]:
train["id_code"] = train["id_code"].apply(lambda x: str(x) + ".png")
test["id_code"] = test["id_code"].apply(lambda x: str(x) + ".png")
train["diagnosis"] = train["diagnosis"].astype(str)



In [6]:
"""
crops black parts around the image (intensity is <= tol)
"""


def crop_image(img, tol=10):
    def crop_image_1(img2d):
        mask = img2d > tol
        return img2d[np.ix_(mask.any(1), mask.any(0))]

    if img.ndim == 2:
        return crop_image_1(img)

    elif img.ndim == 3:
        try:
            img_cpy = img.copy()
            h, w, _ = img.shape
            img1 = cv2.resize(crop_image_1(img[:, :, 0]), (w, h))
            img2 = cv2.resize(crop_image_1(img[:, :, 1]), (w, h))
            img3 = cv2.resize(crop_image_1(img[:, :, 2]), (w, h))

            img[:, :, 0] = img1
            img[:, :, 1] = img2
            img[:, :, 2] = img3

        except Exception:
            return img_cpy

        return img


"""
crops black parts and enhances image (Ben Graham's method)
"""


def preprocess_image(img):
    img = crop_image(img)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0, 0), IMG_SIZE / 10), -4, 128)
    return img




In [7]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.2,
    horizontal_flip=True,
    preprocessing_function=preprocess_image,
)

train_gen = train_datagen.flow_from_dataframe(
    dataframe=train,
    directory=TRAIN_IMG_DIR,
    x_col="id_code",
    y_col="diagnosis",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    target_size=(IMG_SIZE, IMG_SIZE),
    subset="training",
    shuffle=True,
)

val_gen = train_datagen.flow_from_dataframe(
    dataframe=train,
    directory=TRAIN_IMG_DIR,
    x_col="id_code",
    y_col="diagnosis",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    target_size=(IMG_SIZE, IMG_SIZE),
    subset="validation",
    shuffle=True,
)

test_datagen = ImageDataGenerator(
    rescale=1.0 / 255, preprocessing_function=preprocess_image
)

test_gen = test_datagen.flow_from_dataframe(
    dataframe=test,
    directory=TEST_IMG_DIR,
    x_col="id_code",
    batch_size=TEST_BATCH_SIZE,
    class_mode=None,
    target_size=(IMG_SIZE, IMG_SIZE),
    shuffle=False,
)



Found 2636 validated image filenames belonging to 5 classes.
Found 659 validated image filenames belonging to 5 classes.
Found 367 validated image filenames.


In [8]:
MODEL_NAME = "conv1"

NB_WARMUP_EPOCHS = 2
NB_EPOCHS = 30
INITIAL_LR = 1e-3

weights_path_template = os.path.join(
    "weights", "{}_weights.hdf5"
)  # last (or best-only) weights
best_weights_path_template = os.path.join(
    "weights", "{}_best.hdf5"
)  # explicitly best checkpoint
log_path_template = os.path.join("logs", "{}_training_log.csv")



In [9]:
os.makedirs("weights", exist_ok=True)
os.makedirs("logs", exist_ok=True)



In [10]:
"""
ResNet50 based model
"""


def get_resnet50(input_shape, nb_out):
    inputs = Input(shape=input_shape)

    base_model = ResNet50(weights="imagenet", include_top=False, input_tensor=inputs)

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dropout(0.5)(x)

    x = Dense(2048, activation="relu")(x)
    x = Dropout(0.5)(x)

    x = Dense(1024, activation="relu")(x)
    x = Dropout(0.5)(x)

    output = Dense(nb_out, activation="softmax", name="final_output")(x)

    model = Model(inputs, output)
    return model




In [11]:
"""
simple CNN
"""


def get_conv1(input_shape, nb_out):
    inputs = Input(shape=input_shape)

    x = Conv2D(64, (7, 7), activation="relu")(inputs)
    x = MaxPooling2D((2, 2))(x)
    x = BatchNormalization()(x)

    x = Conv2D(64, (7, 7), activation="relu")(x)
    x = MaxPooling2D((2, 2))(x)
    x = BatchNormalization()(x)

    x = Conv2D(128, (5, 5), activation="relu")(x)
    x = MaxPooling2D((2, 2))(x)
    x = BatchNormalization()(x)

    x = Conv2D(256, (3, 3), activation="relu")(x)
    x = MaxPooling2D((2, 2))(x)
    x = BatchNormalization()(x)

    x = Conv2D(512, (3, 3), activation="relu")(x)
    x = MaxPooling2D((2, 2))(x)
    x = BatchNormalization()(x)

    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)

    x = Dense(2048, activation="relu")(x)
    x = Dropout(0.5)(x)

    x = Dense(1024, activation="relu")(x)
    x = Dropout(0.5)(x)

    output = Dense(nb_out, activation="softmax", name="final_output")(x)

    model = Model(inputs, output)
    return model




In [12]:
"""
returns model
"""


def _scan_kaggle_input_for_weights(fnames):
    base = "/kaggle/input"
    out = []
    if not os.path.isdir(base):
        return out
    fnames = set(fnames)
    for root, dirs, files in os.walk(base):
        for f in files:
            if f in fnames:
                out.append(os.path.join(root, f))
    return out


def _candidate_weight_paths(name: str):
    fnames = [
        f"{name}_best.hdf5",
        f"{name}_best.h5",
        f"{name}_weights.hdf5",
        f"{name}_weights.h5",
        f"{name}.hdf5",
        f"{name}.h5",
        f"{name}_weights.keras",
        f"{name}.keras",
    ]

    candidates = []
    for f in fnames:
        candidates.extend(
            [
                os.path.join("weights", f),
                os.path.join("/kaggle/input/aptos-2019-conv1-weights", f),
                os.path.join("../input/aptos-2019-conv1-weights", f),
                os.path.join("/kaggle/input", "aptos-2019-conv1-weights", f),
            ]
        )

    candidates.extend(_scan_kaggle_input_for_weights(fnames))

    def _rank(p):
        base = os.path.basename(p)
        is_local = 0 if os.path.abspath(p).startswith(os.path.abspath("weights")) else 1
        prefer_best = 0 if base in (f"{name}_best.hdf5", f"{name}_best.h5") else 1
        exact = 0 if base == f"{name}_weights.hdf5" else 1
        return (is_local, prefer_best, exact, len(p))

    seen = set()
    out = []
    for p in sorted(set(candidates), key=_rank):
        if p not in seen:
            out.append(p)
            seen.add(p)
    return out


def get_model(name, input_shape, nb_out):
    models = {
        "resnet50": get_resnet50,
        "conv1": get_conv1,
    }

    if name not in models:
        print(f"No model named '{name}'")
        return None

    model = models[name](input_shape, nb_out)

    loaded = False
    tried = []
    for wp in _candidate_weight_paths(name):
        if os.path.isfile(wp):
            tried.append(wp)
            try:
                model.load_weights(wp)
                print(f"loaded model from {wp}")
                loaded = True
                break
            except Exception as e:
                print(f"found weights at {wp} but failed to load: {e}")

    if not loaded:
        msg = "weights not found in any candidate location."
        print(msg)
        cand = _candidate_weight_paths(name)
        print("First 20 candidates:\n" + "\n".join(cand[:20]))

        if (not TRAINING) and (not ALLOW_RANDOM_WEIGHTS_INFERENCE):
            raise RuntimeError(
                "No weights were loaded and TRAINING=False. "
                "Set TRAINING=True to train and create weights, or set "
                "ALLOW_RANDOM_WEIGHTS_INFERENCE=True to force a (likely very low-scoring) submission."
            )

    return model




In [13]:
"""
trains a ResNet50-based model
"""


def train_resnet50(
    model, train_generator, val_generator, weights_path, best_weights_path, log_path
):
    for i in range(len(model.layers)):
        model.layers[i].trainable = False
    for i in range(-5, 0):
        model.layers[i].trainable = True

    metrics_list = ["accuracy"]
    optimizer = Adam(learning_rate=INITIAL_LR)

    model.compile(
        optimizer=optimizer, loss="categorical_crossentropy", metrics=metrics_list
    )

    mc_best = ModelCheckpoint(
        best_weights_path,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    )
    mc_last = ModelCheckpoint(
        weights_path,
        monitor="val_accuracy",
        mode="max",
        save_best_only=False,
        verbose=0,
    )
    cl = CSVLogger(log_path)

    step_size_train = max(1, train_generator.n // train_generator.batch_size)
    step_size_val = max(1, val_generator.n // val_generator.batch_size)

    model.fit(
        train_generator,
        steps_per_epoch=step_size_train,
        validation_data=val_generator,
        validation_steps=step_size_val,
        epochs=NB_WARMUP_EPOCHS,
        callbacks=[mc_best, mc_last, cl],
        verbose=1,
    )

    train_generator.reset()
    val_generator.reset()

    for i in range(len(model.layers)):
        model.layers[i].trainable = True

    optimizer = Adam(learning_rate=INITIAL_LR)
    model.compile(
        optimizer=optimizer, loss="categorical_crossentropy", metrics=metrics_list
    )

    mc_best = ModelCheckpoint(
        best_weights_path,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    )
    mc_last = ModelCheckpoint(
        weights_path,
        monitor="val_accuracy",
        mode="max",
        save_best_only=False,
        verbose=0,
    )
    cl = CSVLogger(log_path)

    # Bugfix: previously this incorrectly used NB_WARMUP_EPOCHS again, preventing full training.
    # This is not a logic rewrite; it's restoring the intended second-stage fine-tune duration.
    model.fit(
        train_generator,
        steps_per_epoch=step_size_train,
        validation_data=val_generator,
        validation_steps=step_size_val,
        epochs=NB_EPOCHS,
        callbacks=[mc_best, mc_last, cl],
        verbose=1,
    )




In [14]:
"""
trains the simple CNN
"""


def train_conv1(
    model, train_generator, val_generator, weights_path, best_weights_path, log_path
):
    metrics_list = ["accuracy"]
    optimizer = Adam(learning_rate=INITIAL_LR)

    model.compile(
        optimizer=optimizer, loss="categorical_crossentropy", metrics=metrics_list
    )

    mc_best = ModelCheckpoint(
        best_weights_path,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    )
    mc_last = ModelCheckpoint(
        weights_path,
        monitor="val_accuracy",
        mode="max",
        save_best_only=False,
        verbose=0,
    )
    cl = CSVLogger(log_path)

    step_size_train = max(1, train_generator.n // train_generator.batch_size)
    step_size_val = max(1, val_generator.n // val_generator.batch_size)

    model.fit(
        train_generator,
        steps_per_epoch=step_size_train,
        validation_data=val_generator,
        validation_steps=step_size_val,
        epochs=NB_EPOCHS,
        callbacks=[mc_best, mc_last, cl],
        verbose=1,
    )




In [15]:
def train_model(name, input_shape, nb_out, train_generator, val_generator):
    model = get_model(name, input_shape, nb_out)

    trainers = {
        "resnet50": train_resnet50,
        "conv1": train_conv1,
    }

    if name not in trainers:
        print(f"No model named '{name}'")
        return

    trainers[name](
        model,
        train_generator,
        val_generator,
        weights_path_template.format(name),
        best_weights_path_template.format(name),
        log_path_template.format(name),
    )




In [16]:
if TRAINING:
    train_model(
        MODEL_NAME, (IMG_SIZE, IMG_SIZE, NB_CHANNELS), NB_CLASSES, train_gen, val_gen
    )

# Prefer best checkpoint for inference if training produced it (or if it exists already).
inference_model = get_model(MODEL_NAME, (IMG_SIZE, IMG_SIZE, NB_CHANNELS), NB_CLASSES)
if inference_model is None:
    raise RuntimeError("Model could not be created; check MODEL_NAME and imports.")

best_wp = best_weights_path_template.format(MODEL_NAME)
last_wp = weights_path_template.format(MODEL_NAME)
if os.path.isfile(best_wp):
    try:
        inference_model.load_weights(best_wp)
        print(f"Using best checkpoint for inference: {best_wp}")
    except Exception as e:
        print(f"Failed to load best checkpoint {best_wp}: {e}")
        if os.path.isfile(last_wp):
            inference_model.load_weights(last_wp)
            print(f"Fell back to last checkpoint for inference: {last_wp}")

test_gen.reset()
preds = inference_model.predict(test_gen, verbose=1)

# Metric-aligned postprocessing (ordinal classes): expected value -> round to nearest class.
# This keeps the same model outputs but generally improves QWK vs hard argmax.
class_ids = np.arange(NB_CLASSES, dtype=np.float32)
expected = (preds * class_ids[None, :]).sum(axis=1)
predictions = np.rint(expected).astype(int)

if len(predictions) != len(test):
    raise RuntimeError(
        f"Prediction length mismatch: got {len(predictions)} preds for {len(test)} test rows"
    )

results = pd.DataFrame(
    {
        "id_code": test["id_code"].str.replace(".png", "", regex=False),
        "diagnosis": predictions,
    }
)

results["id_code"] = results["id_code"].astype(str)
results["diagnosis"] = results["diagnosis"].astype(int).clip(0, NB_CLASSES - 1)

expected_ids = pd.read_csv(TEST_CSV)["id_code"].astype(str).values
if not np.array_equal(results["id_code"].values, expected_ids):
    tmp = results.set_index("id_code").reindex(expected_ids).reset_index()
    if tmp["diagnosis"].isna().any():
        raise RuntimeError(
            "Submission alignment failed: missing predictions for some test ids."
        )
    results = tmp

results.to_csv("submission.csv", index=False)

print(results.head(10))
print("Wrote submission.csv with shape:", results.shape)
print("diagnosis value counts:\n", results["diagnosis"].value_counts().sort_index())
print(
    "Weights candidates searched (top 10):\n"
    + "\n".join(_candidate_weight_paths(MODEL_NAME)[:10])
)

2026-05-10 22:27:13.888152: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778452033.888629      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:c5:00.0, compute capability: 8.6


weights not found in any candidate location.
First 20 candidates:
weights/conv1_best.h5
weights/conv1_best.hdf5
weights/conv1_weights.hdf5
weights/conv1.h5
weights/conv1.hdf5
weights/conv1.keras
weights/conv1_weights.h5
weights/conv1_weights.keras
../input/aptos-2019-conv1-weights/conv1_best.h5
../input/aptos-2019-conv1-weights/conv1_best.hdf5
/kaggle/input/aptos-2019-conv1-weights/conv1_best.h5
/kaggle/input/aptos-2019-conv1-weights/conv1_best.hdf5
../input/aptos-2019-conv1-weights/conv1_weights.hdf5
/kaggle/input/aptos-2019-conv1-weights/conv1_weights.hdf5
../input/aptos-2019-conv1-weights/conv1.h5
../input/aptos-2019-conv1-weights/conv1.hdf5
../input/aptos-2019-conv1-weights/conv1.keras
/kaggle/input/aptos-2019-conv1-weights/conv1.h5
/kaggle/input/aptos-2019-conv1-weights/conv1.hdf5
../input/aptos-2019-conv1-weights/conv1_weights.h5


ValueError: The filepath provided must end in `.keras` (Keras model format). Received: filepath=weights/conv1_best.hdf5